# Seeded runs spike M-info

This notebook computes Gaussian-copula W/M information on recurrent-layer spike trains for all saved seeded runs.

Scope:
- checkpoints: every saved 2-class seeded run checkpoint from `Checkpoints/SeededRuns`
- data source: recurrent spike trains `spk_rec` collected from the SHD test split
- subset sizes: `2, 4, 8, 16, 32`
- outputs: per-model M-info tables plus a set of performance plots

The smoke cell runs the first complete seed pair only. The final cell runs the full analysis when `RUN_FULL` is set to `True`.

In [7]:
from pathlib import Path
import importlib
import io
import warnings
from contextlib import redirect_stdout

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from scipy.stats import norm, rankdata

import seeded_runs_common as seeded_runs_common

seeded_runs_common = importlib.reload(seeded_runs_common)

from wimfo.W_M_Info import W_M_calculator
from wimfo.utils.utils_gauss import get_cov

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120

PROJECT_FILES = seeded_runs_common.PROJECT_FILES
CHECKPOINT_ROOT = seeded_runs_common.CHECKPOINT_ROOT
OUTPUT_DIR = PROJECT_FILES / "seeded_run_spike_minfo_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DISCOVERY_GLOB = "seeded_run_*/2class_lognormal/*.pt"
SUBSET_SIZES = [2, 4, 8, 16, 32]
MAX_BATCHES = 2
SPIKE_DOWNSAMPLE_STRIDE = 4
LAG = 1
RIDGE = 1e-2
SMOKE_MAX_BATCHES = 1
ROLE_ORDER = ["homogeneous_anchor", "heterogeneous_sampled"]
ROLE_LABELS = {
    "homogeneous_anchor": "Homogeneous anchor",
    "heterogeneous_sampled": "Heterogeneous sampled",
}

print(f"Checkpoint root: {CHECKPOINT_ROOT}")
print(f"Output dir      : {OUTPUT_DIR}")
print(f"Discovery glob  : {DISCOVERY_GLOB}")
print(f"Subset sizes    : {SUBSET_SIZES}")
print(f"Max batches     : {MAX_BATCHES}")
print(f"Spike stride    : {SPIKE_DOWNSAMPLE_STRIDE}")
print(f"Lag             : {LAG}")
print(f"Ridge           : {RIDGE}")

Checkpoint root: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\Checkpoints\SeededRuns
Output dir      : C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs
Discovery glob  : seeded_run_*/2class_lognormal/*.pt
Subset sizes    : [2, 4, 8, 16, 32]
Max batches     : 2
Spike stride    : 4
Lag             : 1
Ridge           : 0.01


In [8]:
def discover_seeded_checkpoints():
    rows = []
    for checkpoint_path in sorted(CHECKPOINT_ROOT.glob(DISCOVERY_GLOB)):
        payload = torch.load(checkpoint_path, map_location="cpu")
        if payload.get("task_key") != "2class":
            continue
        if payload.get("task") != "binary_parity":
            raise ValueError(f"Unexpected task in {checkpoint_path.name}: {payload.get('task')}")

        history = payload.get("history", {})
        rows.append({
            "run_label": payload.get("run_label"),
            "pair_seed": int(payload.get("pair_seed")),
            "pair_role": payload.get("pair_role"),
            "pair_role_label": ROLE_LABELS.get(payload.get("pair_role"), payload.get("pair_role")),
            "task_key": payload.get("task_key"),
            "task_name": payload.get("task"),
            "mem_distribution_family": payload.get("mem_distribution_family"),
            "checkpoint": str(checkpoint_path),
            "checkpoint_name": checkpoint_path.name,
            "hidden_tau_syn_ms": float(payload.get("hidden_tau_syn_ms")),
            "hidden_tau_mem_geom_mean_ms": float(payload.get("hidden_tau_mem_geom_mean_ms")),
            "final_test_acc": float(history.get("test_acc", [float("nan")])[-1]),
            "final_test_loss": float(history.get("test_loss", [float("nan")])[-1]),
        })

    table = pd.DataFrame(rows)
    if table.empty:
        raise RuntimeError("No seeded-run 2-class checkpoints were found.")

    table = table.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    return table

checkpoints_df = discover_seeded_checkpoints()

assert checkpoints_df["task_key"].eq("2class").all()
assert checkpoints_df["task_name"].eq("binary_parity").all()
assert checkpoints_df.groupby("pair_seed")["pair_role"].nunique().eq(2).all()

accuracy_df = (
    checkpoints_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
    .reset_index()
    .sort_values("pair_seed")
    .reset_index(drop=True)
)
if set(ROLE_ORDER).issubset(accuracy_df.columns):
    accuracy_df["hetero_minus_hom_acc"] = (
        accuracy_df["heterogeneous_sampled"] - accuracy_df["homogeneous_anchor"]
    )

display(checkpoints_df)
display(accuracy_df)
print(f"Found {len(checkpoints_df)} checkpoints across {checkpoints_df['pair_seed'].nunique()} seeded pairs.")

,run_label,pair_seed,pair_role,pair_role_label,task_key,task_name,mem_distribution_family,checkpoint,checkpoint_name,hidden_tau_syn_ms,hidden_tau_mem_geom_mean_ms,final_test_acc,final_test_loss
0,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,24.687252,0.824205,0.461067
1,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,24.687252,0.826855,0.424700
2,seeded_run_2,201,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed201_het_lognormal.pt,10.0,23.203168,0.864841,0.360071
3,seeded_run_2,201,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed201_hom_geommean_lognormal.pt,10.0,23.203168,0.844965,0.404792
4,seeded_run_1,202,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed202_het_lognormal.pt,10.0,29.308363,0.845406,0.420902
5,seeded_run_1,202,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed202_hom_geommean_lognormal.pt,10.0,29.308363,0.779594,0.510261
6,seeded_run_1,210,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed210_het_lognormal.pt,10.0,22.405558,0.801237,0.478489
7,seeded_run_1,210,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed210_hom_geommean_lognormal.pt,10.0,22.405558,0.730565,0.677380
8,seeded_run_2,211,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed211_het_lognormal.pt,10.0,28.072028,0.506184,0.582101
9,seeded_run_2,211,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed211_hom_geommean_lognormal.pt,10.0,28.072028,0.758834,0.504108


pair_role,pair_seed,heterogeneous_sampled,homogeneous_anchor,hetero_minus_hom_acc
0,101,0.824205,0.826855,-0.002650
1,201,0.864841,0.844965,0.019876
2,202,0.845406,0.779594,0.065813
3,210,0.801237,0.730565,0.070671
4,211,0.506184,0.758834,-0.252650
5,221,0.822880,0.809187,0.013693
6,231,0.762367,0.733657,0.028710
7,241,0.827297,0.758834,0.068463
8,340,0.791519,0.753092,0.038428
9,440,0.810512,0.848498,-0.037986


Found 20 checkpoints across 10 seeded pairs.


In [9]:
print("Loading SHD test cache...")
TEST_CACHE = seeded_runs_common.SHDCache(seeded_runs_common.SHD_TEST)

def is_cache(obj):
    return hasattr(obj, "units") and hasattr(obj, "times") and hasattr(obj, "labels")

def load_seeded_model_from_checkpoint(checkpoint_path):
    payload = torch.load(Path(checkpoint_path), map_location="cpu")
    pair = seeded_runs_common.build_seeded_pair(
        master_seed=int(payload["pair_seed"]),
        task_key=payload["task_key"],
        mem_distribution_family=payload["mem_distribution_family"],
    )

    if payload["pair_role"] == "homogeneous_anchor":
        model = pair["hom_model"]
        prms = dict(pair["hom_prms"])
    elif payload["pair_role"] == "heterogeneous_sampled":
        model = pair["hetero_model"]
        prms = dict(pair["hetero_prms"])
    else:
        raise ValueError(f"Unexpected pair role: {payload['pair_role']}")

    prms.update(payload.get("prms", {}))
    prms["dtype"] = torch.float
    prms["device"] = seeded_runs_common.DEVICE
    prms["cuda"] = seeded_runs_common.DEVICE.type == "cuda"

    model.load_state_dict(payload["model_state_dict"])
    model = model.to(seeded_runs_common.DEVICE)
    model.eval()
    return model, prms, payload

Loading SHD test cache...
  SHDCache: 2264 samples loaded from shd_test.h5


In [ ]:
def gaussian_copula_normalize(data: np.ndarray) -> np.ndarray:
    transformed = np.zeros_like(data, dtype=np.float64)
    for idx, row in enumerate(data):
        if np.allclose(row, row[0]):
            continue
        ranks = rankdata(row, method="average")
        uniform = np.clip((ranks - 0.5) / len(row), 1e-6, 1.0 - 1e-6)
        transformed[idx] = norm.ppf(uniform)
    return transformed

def inject_small_jitter(data: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    row_std = np.nanstd(data, axis=1)
    deg_idx = np.flatnonzero(row_std <= 1e-12)
    if deg_idx.size == 0:
        return data

    base = np.linspace(-1.0, 1.0, data.shape[1], dtype=np.float64)
    for offset, ridx in enumerate(deg_idx, start=1):
        data[ridx] = (eps * offset) * base
    return data

def compute_wm_from_matrix(hidden_data, lag=1, ridge=1e-2):
    gaussian_data = gaussian_copula_normalize(hidden_data)
    gaussian_data = np.nan_to_num(gaussian_data, nan=0.0, posinf=0.0, neginf=0.0)
    gaussian_data = inject_small_jitter(gaussian_data)

    opt_order = [
        ("Adam", {"atol": 1e-3, "rtol": 1e-3, "max_iter": 30000}),
        ("Adam", {"atol": 5e-3, "rtol": 5e-3, "max_iter": 60000}),
        ("Newton", None),
    ]

    lag0 = max(int(lag), 1)
    lag_candidates = sorted(set([lag0, lag0 * 2, lag0 * 4]))
    stride_candidates = [1, 2, 4]

    for stride in stride_candidates:
        data_view = gaussian_data[:, ::stride]
        if data_view.shape[1] <= max(8, 2 * data_view.shape[0] + 2):
            continue
        for lag_try in lag_candidates:
            for optimiser, options in opt_order:
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore", RuntimeWarning)
                        with io.StringIO() as buf, redirect_stdout(buf):
                            w_bits, m_bits = W_M_calculator(
                                data_view,
                                t=lag_try,
                                option="data",
                                type="gaussian",
                                unit="bits",
                                verbose=False,
                                optimiser=optimiser,
                                options=options,
                            )
                except Exception:
                    continue
                if np.isfinite(w_bits) and np.isfinite(m_bits):
                    return {
                        "W_bits": float(w_bits),
                        "M_bits": float(m_bits),
                        "samples": int(data_view.shape[1]),
                        "optimiser": f"{optimiser}(data)",
                        "lag": int(lag_try),
                        "stride": int(stride),
                    }

    ridge_candidates = sorted(set([float(ridge), 5.0 * float(ridge), 1e-1, 2e-1, 5e-1, 1.0]))
    last_error = None

    for stride in stride_candidates:
        data_view = gaussian_data[:, ::stride]
        if data_view.shape[1] <= max(8, 2 * data_view.shape[0] + 2):
            continue

        for lag_try in lag_candidates:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    cov = np.asarray(get_cov(data_view, t=lag_try), dtype=np.float64)
            except Exception as exc:
                last_error = exc
                continue

            cov = np.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0)
            cov = 0.5 * (cov + cov.T)
            scale = np.trace(cov) / max(cov.shape[0], 1)
            if not np.isfinite(scale) or scale <= 0.0:
                scale = 1.0
            eye = np.eye(cov.shape[0], dtype=np.float64)

            for ridge_try in ridge_candidates:
                lagged_cov = cov + ridge_try * scale * eye
                try:
                    eigvals, eigvecs = np.linalg.eigh(lagged_cov)
                    min_eval = max(scale * 1e-8, 1e-10)
                    eigvals = np.clip(eigvals, min_eval, None)
                    lagged_cov = (eigvecs * eigvals[np.newaxis, :]) @ eigvecs.T
                    lagged_cov = 0.5 * (lagged_cov + lagged_cov.T)
                except np.linalg.LinAlgError as exc:
                    last_error = exc
                    continue

                for optimiser, options in opt_order:
                    try:
                        with warnings.catch_warnings():
                            warnings.simplefilter("ignore", RuntimeWarning)
                            with io.StringIO() as buf, redirect_stdout(buf):
                                w_bits, m_bits = W_M_calculator(
                                    lagged_cov,
                                    option="distr",
                                    type="gaussian",
                                    unit="bits",
                                    verbose=False,
                                    optimiser=optimiser,
                                    options=options,
                                )
                    except Exception as exc:
                        last_error = exc
                        continue

                    if np.isfinite(w_bits) and np.isfinite(m_bits):
                        return {
                            "W_bits": float(w_bits),
                            "M_bits": float(m_bits),
                            "samples": int(data_view.shape[1]),
                            "optimiser": f"{optimiser}(distr,ridge={ridge_try:.3g})",
                            "lag": int(lag_try),
                            "stride": int(stride),
                        }

    if last_error is None:
        raise RuntimeError("W/M calculation failed for all optimiser paths.")
    raise RuntimeError(f"W/M calculation failed for all optimiser paths. Last error: {last_error}")

@torch.no_grad()
def collect_all_hidden_spikes(model, prms, data_source, max_batches=2, downsample_stride=4):
    stride = max(int(downsample_stride), 1)
    nb_hidden = int(model.network[0].output_size)
    chunks = []
    ctx = seeded_runs_common.shd_open_cached if is_cache(data_source) else seeded_runs_common.shd_open

    with ctx(data_source) as (units, times, labels):
        for batch_idx, (x, _) in enumerate(
            seeded_runs_common.shd_generator(
                units,
                times,
                labels,
                prms,
                shuffle=False,
                epoch=0,
                drop_last=False,
            )
        ):
            if batch_idx >= max_batches:
                break

            x = x.to(seeded_runs_common.DEVICE)
            layer_recs = model(0, 0, x)
            spikes = layer_recs[0][2][:, ::stride, :].detach().cpu().numpy()
            if not np.all(np.isin(spikes, [0.0, 1.0])):
                raise ValueError("Collected spike tensor is not strictly binary.")
            spikes = np.transpose(spikes, (2, 0, 1)).reshape(nb_hidden, -1)
            chunks.append(spikes)

    if not chunks:
        raise RuntimeError("No batches were collected for spike-train M-info.")
    return np.concatenate(chunks, axis=1)

def compute_spike_m_info_sweep(model, prms, data_source, subset_sizes=None, max_batches=2, lag=1, downsample_stride=4, ridge=1e-2):
    if subset_sizes is None:
        subset_sizes = SUBSET_SIZES

    nb_hidden = int(model.network[0].output_size)
    all_hidden = collect_all_hidden_spikes(
        model,
        prms,
        data_source,
        max_batches=max_batches,
        downsample_stride=downsample_stride,
    )

    results = {}
    for subset_size in subset_sizes:
        subset_size = int(subset_size)
        indices = np.linspace(0, nb_hidden - 1, min(subset_size, nb_hidden), dtype=int)
        try:
            metrics = compute_wm_from_matrix(all_hidden[indices, :], lag=lag, ridge=ridge)
            metrics["subset_size"] = subset_size
            metrics["indices"] = indices.tolist()
            results[subset_size] = metrics
        except Exception as exc:
            results[subset_size] = {
                "subset_size": subset_size,
                "indices": indices.tolist(),
                "W_bits": float("nan"),
                "M_bits": float("nan"),
                "samples": 0,
                "optimiser": "error",
                "lag": int(lag),
                "stride": int(downsample_stride),
                "error": str(exc),
            }
    return results

In [ ]:
def analyse_checkpoint_row(record, data_source, subset_sizes=None, max_batches=2, downsample_stride=4, lag=1, ridge=1e-2):
    model, prms, payload = load_seeded_model_from_checkpoint(record["checkpoint"])
    sweep = compute_spike_m_info_sweep(
        model,
        prms,
        data_source,
        subset_sizes=subset_sizes,
        max_batches=max_batches,
        lag=lag,
        downsample_stride=downsample_stride,
        ridge=ridge,
    )

    rows = []
    for subset_size, metrics in sweep.items():
        row = dict(record)
        row.update({
            "subset_size": int(subset_size),
            "W_bits": metrics.get("W_bits", float("nan")),
            "M_bits": metrics.get("M_bits", float("nan")),
            "samples": int(metrics.get("samples", 0)),
            "lag": int(metrics.get("lag", lag)),
            "stride": int(metrics.get("stride", downsample_stride)),
            "optimiser": metrics.get("optimiser", ""),
            "indices": ",".join(str(idx) for idx in metrics.get("indices", [])),
            "error": metrics.get("error", ""),
            "data_kind": "recurrent_spike_trains",
        })
        rows.append(row)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return rows

def run_spike_minfo_analysis(checkpoint_table, data_source, subset_sizes=None, max_batches=2, downsample_stride=4, lag=1, ridge=1e-2):
    if subset_sizes is None:
        subset_sizes = SUBSET_SIZES

    all_rows = []
    ordered = checkpoint_table.sort_values(["pair_seed", "pair_role"]).reset_index(drop=True)
    for _, record in ordered.iterrows():
        print(f"Running spike-train M-info for seed {record['pair_seed']} :: {record['pair_role']}")
        all_rows.extend(
            analyse_checkpoint_row(
                record.to_dict(),
                data_source=data_source,
                subset_sizes=subset_sizes,
                max_batches=max_batches,
                downsample_stride=downsample_stride,
                lag=lag,
                ridge=ridge,
            )
        )

    results_df = pd.DataFrame(all_rows)
    if results_df.empty:
        raise RuntimeError("The spike-train M-info analysis produced no rows.")
    return results_df.sort_values(["pair_seed", "pair_role", "subset_size"]).reset_index(drop=True)

def summarise_pair_deltas(results_df):
    paired = results_df.pivot_table(
        index=["pair_seed", "subset_size"],
        columns="pair_role",
        values=["M_bits", "W_bits"],
    )
    paired.columns = [f"{metric}_{role}" for metric, role in paired.columns]
    paired = paired.reset_index().sort_values(["pair_seed", "subset_size"]).reset_index(drop=True)

    if {"M_bits_homogeneous_anchor", "M_bits_heterogeneous_sampled"}.issubset(paired.columns):
        paired["hetero_minus_hom_M_bits"] = (
            paired["M_bits_heterogeneous_sampled"] - paired["M_bits_homogeneous_anchor"]
        )
    if {"W_bits_homogeneous_anchor", "W_bits_heterogeneous_sampled"}.issubset(paired.columns):
        paired["hetero_minus_hom_W_bits"] = (
            paired["W_bits_heterogeneous_sampled"] - paired["W_bits_homogeneous_anchor"]
        )
    return paired

def save_analysis_artifacts(results_df, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    clean_results = results_df.sort_values(["pair_seed", "pair_role", "subset_size"]).reset_index(drop=True)
    role_summary_df = (
        clean_results.groupby(["pair_role", "pair_role_label", "subset_size"], as_index=False)
        .agg(
            M_bits_mean=("M_bits", "mean"),
            M_bits_std=("M_bits", "std"),
            W_bits_mean=("W_bits", "mean"),
            W_bits_std=("W_bits", "std"),
            final_test_acc_mean=("final_test_acc", "mean"),
            n_models=("checkpoint", "nunique"),
        )
        .sort_values(["pair_role", "subset_size"])
        .reset_index(drop=True)
    )
    delta_df = summarise_pair_deltas(clean_results)

    checkpoint_meta_df = (
        clean_results[["pair_seed", "pair_role", "pair_role_label", "final_test_acc", "run_label", "mem_distribution_family"]]
        .drop_duplicates()
        .sort_values(["pair_seed", "pair_role"])
        .reset_index(drop=True)
    )
    accuracy_df = (
        checkpoint_meta_df.pivot(index="pair_seed", columns="pair_role", values="final_test_acc")
        .reset_index()
        .sort_values("pair_seed")
        .reset_index(drop=True)
    )
    if set(ROLE_ORDER).issubset(accuracy_df.columns):
        accuracy_df["hetero_minus_hom_acc"] = (
            accuracy_df["heterogeneous_sampled"] - accuracy_df["homogeneous_anchor"]
        )

    paths = {
        "results_csv": output_dir / "seeded_runs_spike_minfo_results.csv",
        "role_summary_csv": output_dir / "seeded_runs_spike_minfo_role_summary.csv",
        "pair_delta_csv": output_dir / "seeded_runs_spike_minfo_pair_delta.csv",
        "accuracy_csv": output_dir / "seeded_runs_spike_accuracy.csv",
        "accuracy_plot": output_dir / "seeded_runs_accuracy_pairs.png",
        "minfo_plot": output_dir / "seeded_runs_spike_minfo_by_subset.png",
        "scatter_plot": output_dir / "seeded_runs_spike_minfo_vs_accuracy.png",
    }

    clean_results.to_csv(paths["results_csv"], index=False)
    role_summary_df.to_csv(paths["role_summary_csv"], index=False)
    delta_df.to_csv(paths["pair_delta_csv"], index=False)
    accuracy_df.to_csv(paths["accuracy_csv"], index=False)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    acc_long = checkpoint_meta_df.copy()
    x_positions = {role: idx for idx, role in enumerate(ROLE_ORDER)}
    for pair_seed in sorted(acc_long["pair_seed"].unique()):
        seed_slice = acc_long[acc_long["pair_seed"] == pair_seed].set_index("pair_role")
        if set(ROLE_ORDER).issubset(seed_slice.index):
            axes[0].plot(
                [x_positions[ROLE_ORDER[0]], x_positions[ROLE_ORDER[1]]],
                [
                    seed_slice.loc[ROLE_ORDER[0], "final_test_acc"],
                    seed_slice.loc[ROLE_ORDER[1], "final_test_acc"],
                ],
                color="0.8",
                linewidth=1,
                zorder=1,
            )
    sns.stripplot(
        data=acc_long,
        x="pair_role",
        y="final_test_acc",
        order=ROLE_ORDER,
        ax=axes[0],
        size=7,
        jitter=0.08,
    )
    axes[0].set_xticks(range(len(ROLE_ORDER)))
    axes[0].set_xticklabels([ROLE_LABELS[role] for role in ROLE_ORDER], rotation=12, ha="right")
    axes[0].set_title("Final test accuracy by model type")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Final test accuracy")

    axes[1].axhline(0.0, color="0.5", linestyle="--", linewidth=1)
    if "hetero_minus_hom_acc" in accuracy_df.columns:
        axes[1].plot(
            accuracy_df["pair_seed"],
            accuracy_df["hetero_minus_hom_acc"],
            marker="o",
            linewidth=1.6,
            color="tab:orange",
        )
    axes[1].set_title("Paired accuracy difference")
    axes[1].set_xlabel("Pair seed")
    axes[1].set_ylabel("Heterogeneous - homogeneous")
    fig.savefig(paths["accuracy_plot"], bbox_inches="tight")
    plt.close(fig)

    plot_df = clean_results.dropna(subset=["M_bits"]).copy()
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    for role in ROLE_ORDER:
        role_df = plot_df[plot_df["pair_role"] == role]
        for pair_seed, seed_slice in role_df.groupby("pair_seed"):
            axes[0].plot(
                seed_slice["subset_size"],
                seed_slice["M_bits"],
                color="0.85",
                linewidth=1,
                zorder=1,
            )
    sns.lineplot(
        data=plot_df,
        x="subset_size",
        y="M_bits",
        hue="pair_role_label",
        style="pair_role_label",
        marker="o",
        dashes=False,
        estimator="mean",
        errorbar="sd",
        ax=axes[0],
    )
    axes[0].set_title("Spike-train M-information by subset size")
    axes[0].set_xlabel("Subset size")
    axes[0].set_ylabel("M-information (bits)")

    axes[1].axhline(0.0, color="0.5", linestyle="--", linewidth=1)
    if "hetero_minus_hom_M_bits" in delta_df.columns:
        for pair_seed, seed_slice in delta_df.groupby("pair_seed"):
            axes[1].plot(
                seed_slice["subset_size"],
                seed_slice["hetero_minus_hom_M_bits"],
                color="0.85",
                linewidth=1,
                zorder=1,
            )
        sns.lineplot(
            data=delta_df,
            x="subset_size",
            y="hetero_minus_hom_M_bits",
            marker="o",
            estimator="mean",
            errorbar="sd",
            color="black",
            ax=axes[1],
        )
    axes[1].set_title("Paired spike-train M-information delta")
    axes[1].set_xlabel("Subset size")
    axes[1].set_ylabel("Heterogeneous - homogeneous (bits)")
    fig.savefig(paths["minfo_plot"], bbox_inches="tight")
    plt.close(fig)

    scatter_df = plot_df.copy()
    scatter_df["subset_label"] = scatter_df["subset_size"].astype(str)
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    sns.scatterplot(
        data=scatter_df,
        x="final_test_acc",
        y="M_bits",
        hue="pair_role_label",
        style="subset_label",
        s=110,
        ax=ax,
    )
    ax.set_title("Final test accuracy vs spike-train M-information")
    ax.set_xlabel("Final test accuracy")
    ax.set_ylabel("M-information (bits)")
    fig.savefig(paths["scatter_plot"], bbox_inches="tight")
    plt.close(fig)

    return role_summary_df, delta_df, accuracy_df, paths

In [ ]:
def build_visual_suite(results_df, delta_df, accuracy_df, output_dir, subset_focus=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if subset_focus is None:
        subset_focus = int(results_df["subset_size"].max())

    role_palette = {
        ROLE_LABELS["homogeneous_anchor"]: "#4C72B0",
        ROLE_LABELS["heterogeneous_sampled"]: "#DD8452",
    }

    plot_df = results_df.dropna(subset=["M_bits", "W_bits", "final_test_acc"]).copy()
    plot_df["subset_size"] = plot_df["subset_size"].astype(int)

    pair_meta_df = (
        plot_df[["pair_seed", "pair_role", "pair_role_label", "final_test_acc"]]
        .drop_duplicates()
        .sort_values(["pair_seed", "pair_role"])
        .reset_index(drop=True)
    )
    subset_focus_df = (
        plot_df[plot_df["subset_size"] == subset_focus]
        .sort_values(["pair_seed", "pair_role"])
        .reset_index(drop=True)
    )
    if subset_focus_df.empty:
        raise RuntimeError(f"No rows are available for subset size {subset_focus}.")

    if "hetero_minus_hom_acc" not in accuracy_df.columns:
        if set(ROLE_ORDER).issubset(accuracy_df.columns):
            accuracy_df = accuracy_df.copy()
            accuracy_df["hetero_minus_hom_acc"] = (
                accuracy_df["heterogeneous_sampled"] - accuracy_df["homogeneous_anchor"]
            )
        else:
            raise RuntimeError("accuracy_df is missing paired accuracy columns.")

    delta_plot_df = delta_df.sort_values(["pair_seed", "subset_size"]).reset_index(drop=True).copy()
    delta_relationship_df = delta_plot_df.merge(
        accuracy_df[["pair_seed", "hetero_minus_hom_acc"]],
        on="pair_seed",
        how="left",
    )
    delta_focus_df = delta_relationship_df[delta_relationship_df["subset_size"] == subset_focus].copy()

    pair_seeds = sorted(pair_meta_df["pair_seed"].unique())
    seed_positions = {seed: idx for idx, seed in enumerate(pair_seeds)}
    ordered_subset_sizes = sorted(plot_df["subset_size"].unique())

    paths = {}

    fig, axes = plt.subplots(1, 2, figsize=(16, 9), sharey=True, constrained_layout=True)
    for pair_seed in pair_seeds:
        y_pos = seed_positions[pair_seed]

        acc_slice = (
            pair_meta_df[pair_meta_df["pair_seed"] == pair_seed]
            .set_index("pair_role")
            .reindex(ROLE_ORDER)
        )
        if acc_slice["final_test_acc"].notna().all():
            axes[0].plot(acc_slice["final_test_acc"], [y_pos, y_pos], color="0.8", linewidth=1.25, zorder=1)
        for role, row in acc_slice.iterrows():
            if pd.notna(row["final_test_acc"]):
                axes[0].scatter(
                    row["final_test_acc"],
                    y_pos,
                    color=role_palette[row["pair_role_label"]],
                    s=80,
                    zorder=2,
                )

        minfo_slice = (
            subset_focus_df[subset_focus_df["pair_seed"] == pair_seed]
            .set_index("pair_role")
            .reindex(ROLE_ORDER)
        )
        if minfo_slice["M_bits"].notna().all():
            axes[1].plot(minfo_slice["M_bits"], [y_pos, y_pos], color="0.8", linewidth=1.25, zorder=1)
        for role, row in minfo_slice.iterrows():
            if pd.notna(row["M_bits"]):
                axes[1].scatter(
                    row["M_bits"],
                    y_pos,
                    color=role_palette[row["pair_role_label"]],
                    s=80,
                    zorder=2,
                )

    for ax in axes:
        ax.set_yticks(range(len(pair_seeds)))
        ax.set_yticklabels([str(seed) for seed in pair_seeds])
        ax.grid(axis="x", alpha=0.25)

    axes[0].set_title("Paired task accuracy by seeded pair")
    axes[0].set_xlabel("Final test accuracy")
    axes[0].set_ylabel("Pair seed")
    axes[1].set_title(f"Paired M-information at subset size {subset_focus}")
    axes[1].set_xlabel("M-information (bits)")
    for label, color in role_palette.items():
        axes[1].scatter([], [], color=color, label=label, s=80)
    axes[1].legend(title="", loc="lower right")

    paths["pairwise_compare_plot"] = output_dir / "seeded_runs_pairwise_accuracy_minfo.png"
    fig.savefig(paths["pairwise_compare_plot"], bbox_inches="tight")
    plt.show()
    plt.close(fig)

    hom_heatmap = (
        plot_df[plot_df["pair_role"] == "homogeneous_anchor"]
        .pivot(index="pair_seed", columns="subset_size", values="M_bits")
        .reindex(index=pair_seeds, columns=ordered_subset_sizes)
    )
    hetero_heatmap = (
        plot_df[plot_df["pair_role"] == "heterogeneous_sampled"]
        .pivot(index="pair_seed", columns="subset_size", values="M_bits")
        .reindex(index=pair_seeds, columns=ordered_subset_sizes)
    )
    delta_heatmap = (
        delta_plot_df.pivot(index="pair_seed", columns="subset_size", values="hetero_minus_hom_M_bits")
        .reindex(index=pair_seeds, columns=ordered_subset_sizes)
    )
    m_value_floor = float(min(hom_heatmap.min().min(), hetero_heatmap.min().min()))
    m_value_ceiling = float(max(hom_heatmap.max().max(), hetero_heatmap.max().max()))

    fig, axes = plt.subplots(1, 3, figsize=(20, 7), constrained_layout=True)
    sns.heatmap(
        hom_heatmap,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        vmin=m_value_floor,
        vmax=m_value_ceiling,
        cbar_kws={"label": "M-information (bits)"},
        ax=axes[0],
    )
    axes[0].set_title("Homogeneous M-information heatmap")
    axes[0].set_xlabel("Subset size")
    axes[0].set_ylabel("Pair seed")

    sns.heatmap(
        hetero_heatmap,
        annot=True,
        fmt=".2f",
        cmap="Oranges",
        vmin=m_value_floor,
        vmax=m_value_ceiling,
        cbar_kws={"label": "M-information (bits)"},
        ax=axes[1],
    )
    axes[1].set_title("Heterogeneous M-information heatmap")
    axes[1].set_xlabel("Subset size")
    axes[1].set_ylabel("")

    sns.heatmap(
        delta_heatmap,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0.0,
        cbar_kws={"label": "Heterogeneous - homogeneous (bits)"},
        ax=axes[2],
    )
    axes[2].set_title("Paired M-information delta heatmap")
    axes[2].set_xlabel("Subset size")
    axes[2].set_ylabel("")

    paths["heatmap_plot"] = output_dir / "seeded_runs_pairwise_minfo_heatmaps.png"
    fig.savefig(paths["heatmap_plot"], bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
    for pair_seed in pair_seeds:
        pair_slice = (
            subset_focus_df[subset_focus_df["pair_seed"] == pair_seed]
            .set_index("pair_role")
            .reindex(ROLE_ORDER)
        )
        if pair_slice[["final_test_acc", "M_bits"]].notna().all().all():
            axes[0].plot(pair_slice["final_test_acc"], pair_slice["M_bits"], color="0.85", linewidth=1.1, zorder=1)

    sns.scatterplot(
        data=subset_focus_df,
        x="final_test_acc",
        y="M_bits",
        hue="pair_role_label",
        palette=role_palette,
        s=120,
        ax=axes[0],
    )
    for row in subset_focus_df.itertuples():
        axes[0].text(row.final_test_acc + 0.0015, row.M_bits + 0.015, str(int(row.pair_seed)), fontsize=8, alpha=0.75)
    axes[0].set_title(f"Accuracy vs M-information at subset size {subset_focus}")
    axes[0].set_xlabel("Final test accuracy")
    axes[0].set_ylabel("M-information (bits)")

    axes[1].axhline(0.0, color="0.5", linestyle="--", linewidth=1)
    axes[1].axvline(0.0, color="0.5", linestyle="--", linewidth=1)
    sns.scatterplot(
        data=delta_relationship_df,
        x="hetero_minus_hom_acc",
        y="hetero_minus_hom_M_bits",
        hue="subset_size",
        palette="viridis",
        s=100,
        ax=axes[1],
    )
    for row in delta_focus_df.itertuples():
        axes[1].text(
            row.hetero_minus_hom_acc + 0.0015,
            row.hetero_minus_hom_M_bits + 0.015,
            str(int(row.pair_seed)),
            fontsize=8,
            alpha=0.75,
        )
    axes[1].set_title("Paired accuracy delta vs M-information delta")
    axes[1].set_xlabel("Heterogeneous - homogeneous accuracy")
    axes[1].set_ylabel("Heterogeneous - homogeneous M bits")
    axes[1].legend(title="Subset size")

    paths["relationship_plot"] = output_dir / "seeded_runs_accuracy_minfo_relationships.png"
    fig.savefig(paths["relationship_plot"], bbox_inches="tight")
    plt.show()
    plt.close(fig)

    fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
    x_positions = {role: idx for idx, role in enumerate(ROLE_ORDER)}
    for pair_seed in pair_seeds:
        acc_slice = (
            pair_meta_df[pair_meta_df["pair_seed"] == pair_seed]
            .set_index("pair_role")
            .reindex(ROLE_ORDER)
        )
        if acc_slice["final_test_acc"].notna().all():
            axes[0, 0].plot(
                [x_positions[ROLE_ORDER[0]], x_positions[ROLE_ORDER[1]]],
                acc_slice["final_test_acc"],
                color="0.8",
                linewidth=1,
                zorder=1,
            )
        for role, row in acc_slice.iterrows():
            if pd.notna(row["final_test_acc"]):
                axes[0, 0].scatter(
                    x_positions[role],
                    row["final_test_acc"],
                    color=role_palette[row["pair_role_label"]],
                    s=75,
                    zorder=2,
                )
    axes[0, 0].set_xticks(range(len(ROLE_ORDER)))
    axes[0, 0].set_xticklabels([ROLE_LABELS[role] for role in ROLE_ORDER], rotation=12, ha="right")
    axes[0, 0].set_title("Paired accuracy summary")
    axes[0, 0].set_xlabel("")
    axes[0, 0].set_ylabel("Final test accuracy")

    sns.lineplot(
        data=plot_df,
        x="subset_size",
        y="M_bits",
        hue="pair_role_label",
        palette=role_palette,
        estimator="mean",
        errorbar="sd",
        marker="o",
        ax=axes[0, 1],
    )
    axes[0, 1].set_title("Mean M-information by subset size")
    axes[0, 1].set_xlabel("Subset size")
    axes[0, 1].set_ylabel("M-information (bits)")
    axes[0, 1].legend(title="")

    sns.lineplot(
        data=plot_df,
        x="subset_size",
        y="W_bits",
        hue="pair_role_label",
        palette=role_palette,
        estimator="mean",
        errorbar="sd",
        marker="o",
        ax=axes[1, 0],
    )
    axes[1, 0].set_title("Mean W-information by subset size")
    axes[1, 0].set_xlabel("Subset size")
    axes[1, 0].set_ylabel("W-information (bits)")
    if axes[1, 0].legend_ is not None:
        axes[1, 0].legend_.remove()

    axes[1, 1].axhline(0.0, color="0.5", linestyle="--", linewidth=1)
    axes[1, 1].axvline(0.0, color="0.5", linestyle="--", linewidth=1)
    sns.scatterplot(
        data=delta_focus_df,
        x="hetero_minus_hom_acc",
        y="hetero_minus_hom_M_bits",
        s=120,
        color="#2A9D8F",
        ax=axes[1, 1],
    )
    for row in delta_focus_df.itertuples():
        axes[1, 1].text(
            row.hetero_minus_hom_acc + 0.0015,
            row.hetero_minus_hom_M_bits + 0.015,
            str(int(row.pair_seed)),
            fontsize=8,
            alpha=0.75,
        )
    axes[1, 1].set_title(f"Unified delta view at subset size {subset_focus}")
    axes[1, 1].set_xlabel("Heterogeneous - homogeneous accuracy")
    axes[1, 1].set_ylabel("Heterogeneous - homogeneous M bits")

    paths["dashboard_plot"] = output_dir / "seeded_runs_unified_dashboard.png"
    fig.savefig(paths["dashboard_plot"], bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return paths

In [6]:
smoke_seed = int(checkpoints_df["pair_seed"].min())
smoke_checkpoints_df = checkpoints_df[checkpoints_df["pair_seed"] == smoke_seed].copy()

smoke_results_df = run_spike_minfo_analysis(
    smoke_checkpoints_df,
    data_source=TEST_CACHE,
    subset_sizes=SUBSET_SIZES,
    max_batches=SMOKE_MAX_BATCHES,
    downsample_stride=SPIKE_DOWNSAMPLE_STRIDE,
    lag=LAG,
    ridge=RIDGE,
)
smoke_role_summary_df, smoke_delta_df, smoke_accuracy_df, smoke_paths = save_analysis_artifacts(
    smoke_results_df,
    output_dir=OUTPUT_DIR / "smoke",
)

display(smoke_results_df)
display(smoke_delta_df)
display(smoke_accuracy_df)
print("Smoke outputs saved:")
for name, path in smoke_paths.items():
    print(f"  {name}: {path}")

Running spike-train M-info for seed 101 :: heterogeneous_sampled


C:\Users\Priya\Desktop\research project (SNN Info Theory)\wimfo\wimfo\utils\utils_gauss.py:44: RuntimeWarning: invalid value encountered in scalar subtract
  mi = 0.5 * (slogdet(Sx)[1] + slogdet(Sy)[1] - slogdet(cov)[1]) / np.log(2)


Running spike-train M-info for seed 101 :: homogeneous_anchor


C:\Users\Priya\AppData\Local\Temp\ipykernel_31228\1366379074.py:157: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[0].set_xticklabels([ROLE_LABELS[role] for role in ROLE_ORDER], rotation=12, ha="right")


,run_label,pair_seed,pair_role,pair_role_label,task_key,task_name,mem_distribution_family,checkpoint,checkpoint_name,hidden_tau_syn_ms,...,subset_size,W_bits,M_bits,samples,lag,stride,optimiser,indices,error,data_kind
0,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,2,0.151327,-0.000032,64000,1,1,Adam(data),"0,31",,recurrent_spike_trains
1,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,4,3.289323,-0.001293,64000,1,1,"Adam(distr,ridge=0.01)","0,10,20,31",,recurrent_spike_trains
2,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,8,3.681236,0.502665,64000,1,1,"Adam(distr,ridge=0.01)","0,4,8,13,17,22,26,31",,recurrent_spike_trains
3,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,16,3.857563,1.219126,64000,1,1,"Adam(distr,ridge=0.05)","0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,31",,recurrent_spike_trains
4,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,32,4.656729,2.443441,64000,1,1,"Adam(distr,ridge=0.05)","0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18...",,recurrent_spike_trains
5,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,2,2.868907,-0.002814,64000,1,1,"Adam(distr,ridge=0.01)","0,31",,recurrent_spike_trains
6,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,4,3.522400,-0.003636,64000,1,1,"Adam(distr,ridge=0.01)","0,10,20,31",,recurrent_spike_trains
7,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,8,2.965927,1.150607,64000,1,1,"Adam(distr,ridge=0.01)","0,4,8,13,17,22,26,31",,recurrent_spike_trains
8,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,16,3.755809,1.279000,64000,1,1,"Adam(distr,ridge=0.01)","0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,31",,recurrent_spike_trains
9,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,32,3.055270,2.399046,64000,1,1,"Adam(distr,ridge=0.05)","0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18...",,recurrent_spike_trains


,pair_seed,subset_size,M_bits_heterogeneous_sampled,M_bits_homogeneous_anchor,W_bits_heterogeneous_sampled,W_bits_homogeneous_anchor,hetero_minus_hom_M_bits,hetero_minus_hom_W_bits
0,101,2,-0.000032,-0.002814,0.151327,2.868907,0.002782,-2.717580
1,101,4,-0.001293,-0.003636,3.289323,3.522400,0.002343,-0.233077
2,101,8,0.502665,1.150607,3.681236,2.965927,-0.647942,0.715309
3,101,16,1.219126,1.279000,3.857563,3.755809,-0.059874,0.101753
4,101,32,2.443441,2.399046,4.656729,3.055270,0.044395,1.601458


pair_role,pair_seed,heterogeneous_sampled,homogeneous_anchor,hetero_minus_hom_acc
0,101,0.824205,0.826855,-0.00265


Smoke outputs saved:
  results_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\smoke\seeded_runs_spike_minfo_results.csv
  role_summary_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\smoke\seeded_runs_spike_minfo_role_summary.csv
  pair_delta_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\smoke\seeded_runs_spike_minfo_pair_delta.csv
  accuracy_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\smoke\seeded_runs_spike_accuracy.csv
  accuracy_plot: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\smoke\seeded_runs_accuracy_pairs.png
  minfo_plot: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\smoke\seeded_runs_spike_minfo_by_subset.png
  scatter

In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_results_df = run_spike_minfo_analysis(
        checkpoints_df,
        data_source=TEST_CACHE,
        subset_sizes=SUBSET_SIZES,
        max_batches=MAX_BATCHES,
        downsample_stride=SPIKE_DOWNSAMPLE_STRIDE,
        lag=LAG,
        ridge=RIDGE,
    )
    full_role_summary_df, full_delta_df, full_accuracy_df, full_paths = save_analysis_artifacts(
        full_results_df,
        output_dir=OUTPUT_DIR,
    )
    full_visual_paths = build_visual_suite(
        full_results_df,
        full_delta_df,
        full_accuracy_df,
        OUTPUT_DIR,
        subset_focus=max(SUBSET_SIZES),
    )

    display(full_role_summary_df)
    display(full_delta_df)
    display(full_accuracy_df)
    print("Full outputs saved:")
    for name, path in full_paths.items():
        print(f"  {name}: {path}")
    print("Visual suite saved:")
    for name, path in full_visual_paths.items():
        print(f"  {name}: {path}")
else:
    print("Set RUN_FULL = True and rerun this cell to process all seeded-run checkpoints.")

## Observed discrete projection scan

This section reuses the delayed binary discrete estimator path from the diagnostics notebook, but only for observed discrete W and M summaries.

Because `wimfo`'s discrete API only accepts 2-neuron delayed projections, each subset is summarised from sampled 2-neuron projections inside that subset. The outputs here mirror the Gaussian notebook's observed W/M tables and visuals without any null or Z-score step.

In [12]:
import seeded_runs_discrete_common as seeded_runs_discrete_common

seeded_runs_discrete_common = importlib.reload(seeded_runs_discrete_common)

DISCRETE_OUTPUT_DIR = OUTPUT_DIR / "discrete_observed_projection_scan"
DISCRETE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DISCRETE_DELAY_TARGET_MS = 15.0
DISCRETE_OBS_SMOKE_SUBSET_SAMPLE_SIZE = 3
DISCRETE_OBS_SMOKE_PAIR_SAMPLE_SIZE = 3
DISCRETE_OBS_FULL_SUBSET_SAMPLE_SIZE = 6
DISCRETE_OBS_FULL_PAIR_SAMPLE_SIZE = 4

delay_probe_payload = torch.load(Path(checkpoints_df.iloc[0]["checkpoint"]), map_location="cpu")
delay_probe_raw_step_ms = float(delay_probe_payload["prms"]["time_step"]) * 1000.0
delay_probe_effective_step_ms = delay_probe_raw_step_ms * SPIKE_DOWNSAMPLE_STRIDE
delay_probe_steps = max(1, int(round(DISCRETE_DELAY_TARGET_MS / delay_probe_effective_step_ms)))
delay_probe_actual_ms = delay_probe_steps * delay_probe_effective_step_ms

print(f"Discrete output dir         : {DISCRETE_OUTPUT_DIR}")
print(f"Discrete delay target       : {DISCRETE_DELAY_TARGET_MS:.1f} ms")
print(f"Discrete effective step     : {delay_probe_effective_step_ms:.3f} ms")
print(f"Discrete delay steps        : {delay_probe_steps}")
print(f"Discrete actual delay       : {delay_probe_actual_ms:.1f} ms")
print(f"Smoke subsets per size      : {DISCRETE_OBS_SMOKE_SUBSET_SAMPLE_SIZE}")
print(f"Smoke pairs per subset      : {DISCRETE_OBS_SMOKE_PAIR_SAMPLE_SIZE}")
print(f"Full subsets per size       : {DISCRETE_OBS_FULL_SUBSET_SAMPLE_SIZE}")
print(f"Full pairs per subset       : {DISCRETE_OBS_FULL_PAIR_SAMPLE_SIZE}")

Discrete output dir         : C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\discrete_observed_projection_scan
Discrete delay target       : 15.0 ms
Discrete effective step     : 4.000 ms
Discrete delay steps        : 4
Discrete actual delay       : 16.0 ms
Smoke subsets per size      : 3
Smoke pairs per subset      : 3
Full subsets per size       : 6
Full pairs per subset       : 4


In [13]:
discrete_smoke_seed = int(checkpoints_df["pair_seed"].min())
discrete_smoke_checkpoints_df = checkpoints_df[checkpoints_df["pair_seed"] == discrete_smoke_seed].copy()

(
    discrete_obs_smoke_pair_df,
    discrete_obs_smoke_subset_df,
    discrete_obs_smoke_model_df,
) = seeded_runs_discrete_common.run_discrete_observed_scan(
    discrete_smoke_checkpoints_df,
    data_source=TEST_CACHE,
    subset_sizes=SUBSET_SIZES,
    subset_sample_size=DISCRETE_OBS_SMOKE_SUBSET_SAMPLE_SIZE,
    pair_sample_size=DISCRETE_OBS_SMOKE_PAIR_SAMPLE_SIZE,
    max_batches=SMOKE_MAX_BATCHES,
    downsample_stride=SPIKE_DOWNSAMPLE_STRIDE,
    delay_target_ms=DISCRETE_DELAY_TARGET_MS,
    role_order=ROLE_ORDER,
    role_labels=ROLE_LABELS,
    label="seeded_runs_discrete_observed_smoke",
)
(
    discrete_obs_smoke_role_summary_df,
    discrete_obs_smoke_delta_df,
    discrete_obs_smoke_accuracy_df,
    discrete_obs_smoke_paths,
) = seeded_runs_discrete_common.save_observed_analysis_artifacts(
    discrete_obs_smoke_pair_df,
    discrete_obs_smoke_subset_df,
    discrete_obs_smoke_model_df,
    output_dir=DISCRETE_OUTPUT_DIR / "smoke",
    role_order=ROLE_ORDER,
    role_labels=ROLE_LABELS,
)
discrete_obs_smoke_visual_paths = seeded_runs_discrete_common.build_observed_visual_suite(
    discrete_obs_smoke_model_df,
    discrete_obs_smoke_delta_df,
    discrete_obs_smoke_accuracy_df,
    output_dir=DISCRETE_OUTPUT_DIR / "smoke",
    role_order=ROLE_ORDER,
    role_labels=ROLE_LABELS,
    subset_focus=max(SUBSET_SIZES),
)

display(discrete_obs_smoke_model_df)
display(discrete_obs_smoke_delta_df)
display(discrete_obs_smoke_role_summary_df)
print("Observed discrete smoke outputs saved:")
for name, path in discrete_obs_smoke_paths.items():
    print(f"  {name}: {path}")
print("Observed discrete smoke visual suite saved:")
for name, path in discrete_obs_smoke_visual_paths.items():
    print(f"  {name}: {path}")

Running observed discrete scan for seed 101 :: heterogeneous_sampled
  k= 2 | sampled   3/496 subsets | mean subset M=  0.0011 W=  0.1833
  k= 4 | sampled   3/35960 subsets | mean subset M=  0.0001 W=  0.0697
  k= 8 | sampled   3/10518300 subsets | mean subset M=  0.0076 W=  0.2846
  k=16 | sampled   3/601080390 subsets | mean subset M=  0.0220 W=  0.1681
  k=32 | sampled   1/1 subsets | mean subset M=  0.0358 W=  0.2183
Running observed discrete scan for seed 101 :: homogeneous_anchor
  k= 2 | sampled   3/496 subsets | mean subset M=  0.0013 W=  0.1759
  k= 4 | sampled   3/35960 subsets | mean subset M=  0.0000 W=  0.0218
  k= 8 | sampled   3/10518300 subsets | mean subset M=  0.0010 W=  0.0114
  k=16 | sampled   3/601080390 subsets | mean subset M=  0.0115 W=  0.1148
  k=32 | sampled   1/1 subsets | mean subset M= -0.0000 W=  0.1118


,run_label,pair_seed,pair_role,pair_role_label,task_key,task_name,mem_distribution_family,checkpoint,checkpoint_name,hidden_tau_syn_ms,...,sampled_pairs_mean,subset_pair_M_mean,subset_pair_M_std,subset_pair_W_mean,subset_pair_W_std,subset_pair_TDMI_mean,valid_pair_count,error_pair_count,delay_steps,delay_ms
0,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,1.0,1.133443e-03,0.001725,0.183259,0.152038,0.184392,1,0,4,16.0
1,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,3.0,1.431870e-04,0.000248,0.069664,0.097003,0.069807,3,0,4,16.0
2,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,3.0,7.613468e-03,0.012163,0.284620,0.115016,0.292233,3,0,4,16.0
3,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,3.0,2.198331e-02,0.033251,0.168137,0.024541,0.190120,3,0,4,16.0
4,seeded_run_1,101,heterogeneous_sampled,Heterogeneous sampled,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_het_lognormal.pt,10.0,...,3.0,3.582327e-02,NaN,0.218262,NaN,0.254085,3,0,4,16.0
5,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,1.0,1.253945e-03,0.001890,0.175904,0.122770,0.177158,1,0,4,16.0
6,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,3.0,2.181707e-05,0.000038,0.021815,0.025065,0.021837,3,0,4,16.0
7,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,3.0,9.915000e-04,0.000823,0.011422,0.010291,0.012414,3,0,4,16.0
8,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,3.0,1.147816e-02,0.015602,0.114773,0.115192,0.126251,3,0,4,16.0
9,seeded_run_1,101,homogeneous_anchor,Homogeneous anchor,2class,binary_parity,lognormal,C:\Users\Priya\Desktop\research project (SNN I...,2class_seed101_hom_geommean_lognormal.pt,10.0,...,3.0,-6.467904e-07,NaN,0.111811,NaN,0.111810,3,0,4,16.0


,pair_seed,subset_size,subset_pair_M_mean_heterogeneous_sampled,subset_pair_M_mean_homogeneous_anchor,subset_pair_W_mean_heterogeneous_sampled,subset_pair_W_mean_homogeneous_anchor,hetero_minus_hom_subset_pair_M_mean,hetero_minus_hom_subset_pair_W_mean
0,101,2,0.001133,1.253945e-03,0.183259,0.175904,-0.000121,0.007355
1,101,4,0.000143,2.181707e-05,0.069664,0.021815,0.000121,0.047849
2,101,8,0.007613,9.915000e-04,0.284620,0.011422,0.006622,0.273197
3,101,16,0.021983,1.147816e-02,0.168137,0.114773,0.010505,0.053364
4,101,32,0.035823,-6.467904e-07,0.218262,0.111811,0.035824,0.106452


,pair_role,pair_role_label,subset_size,subset_pair_M_mean_mean,subset_pair_M_mean_std,subset_pair_W_mean_mean,subset_pair_W_mean_std,subset_pair_TDMI_mean_mean,final_test_acc_mean,n_models
0,heterogeneous_sampled,Heterogeneous sampled,2,1.133443e-03,NaN,0.183259,NaN,0.184392,0.824205,1
1,heterogeneous_sampled,Heterogeneous sampled,4,1.431870e-04,NaN,0.069664,NaN,0.069807,0.824205,1
2,heterogeneous_sampled,Heterogeneous sampled,8,7.613468e-03,NaN,0.284620,NaN,0.292233,0.824205,1
3,heterogeneous_sampled,Heterogeneous sampled,16,2.198331e-02,NaN,0.168137,NaN,0.190120,0.824205,1
4,heterogeneous_sampled,Heterogeneous sampled,32,3.582327e-02,NaN,0.218262,NaN,0.254085,0.824205,1
5,homogeneous_anchor,Homogeneous anchor,2,1.253945e-03,NaN,0.175904,NaN,0.177158,0.826855,1
6,homogeneous_anchor,Homogeneous anchor,4,2.181707e-05,NaN,0.021815,NaN,0.021837,0.826855,1
7,homogeneous_anchor,Homogeneous anchor,8,9.915000e-04,NaN,0.011422,NaN,0.012414,0.826855,1
8,homogeneous_anchor,Homogeneous anchor,16,1.147816e-02,NaN,0.114773,NaN,0.126251,0.826855,1
9,homogeneous_anchor,Homogeneous anchor,32,-6.467904e-07,NaN,0.111811,NaN,0.111810,0.826855,1


Observed discrete smoke outputs saved:
  pair_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\discrete_observed_projection_scan\smoke\seeded_runs_discrete_observed_pair_records.csv
  subset_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\discrete_observed_projection_scan\smoke\seeded_runs_discrete_observed_subset_records.csv
  model_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\discrete_observed_projection_scan\smoke\seeded_runs_discrete_observed_model_summary.csv
  role_summary_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\discrete_observed_projection_scan\smoke\seeded_runs_discrete_observed_role_summary.csv
  pair_delta_csv: C:\Users\Priya\Desktop\research project (SNN Info Theory)\Project Files\seeded_run_spike_minfo_outputs\discrete_observed_

In [ ]:
RUN_DISCRETE_OBS_FULL = False

if RUN_DISCRETE_OBS_FULL:
    (
        full_discrete_obs_pair_df,
        full_discrete_obs_subset_df,
        full_discrete_obs_model_df,
    ) = seeded_runs_discrete_common.run_discrete_observed_scan(
        checkpoints_df,
        data_source=TEST_CACHE,
        subset_sizes=SUBSET_SIZES,
        subset_sample_size=DISCRETE_OBS_FULL_SUBSET_SAMPLE_SIZE,
        pair_sample_size=DISCRETE_OBS_FULL_PAIR_SAMPLE_SIZE,
        max_batches=MAX_BATCHES,
        downsample_stride=SPIKE_DOWNSAMPLE_STRIDE,
        delay_target_ms=DISCRETE_DELAY_TARGET_MS,
        role_order=ROLE_ORDER,
        role_labels=ROLE_LABELS,
        label="seeded_runs_discrete_observed_full",
    )
    (
        full_discrete_obs_role_summary_df,
        full_discrete_obs_delta_df,
        full_discrete_obs_accuracy_df,
        full_discrete_obs_paths,
    ) = seeded_runs_discrete_common.save_observed_analysis_artifacts(
        full_discrete_obs_pair_df,
        full_discrete_obs_subset_df,
        full_discrete_obs_model_df,
        output_dir=DISCRETE_OUTPUT_DIR,
        role_order=ROLE_ORDER,
        role_labels=ROLE_LABELS,
    )
    full_discrete_obs_visual_paths = seeded_runs_discrete_common.build_observed_visual_suite(
        full_discrete_obs_model_df,
        full_discrete_obs_delta_df,
        full_discrete_obs_accuracy_df,
        output_dir=DISCRETE_OUTPUT_DIR,
        role_order=ROLE_ORDER,
        role_labels=ROLE_LABELS,
        subset_focus=max(SUBSET_SIZES),
    )

    display(full_discrete_obs_model_df)
    display(full_discrete_obs_delta_df)
    display(full_discrete_obs_role_summary_df)
    print("Full observed discrete outputs saved:")
    for name, path in full_discrete_obs_paths.items():
        print(f"  {name}: {path}")
    print("Full observed discrete visual suite saved:")
    for name, path in full_discrete_obs_visual_paths.items():
        print(f"  {name}: {path}")
else:
    print("Set RUN_DISCRETE_OBS_FULL = True and rerun this cell to process all seeded-run checkpoints with the observed discrete estimator.")